# 10s30s vs ERZ7 — correlation & hedge-ratio monitor

Measures how much the **10s30s curve** moves with **ERZ7** (3M Euribor
Dec-2027 future). Both the **EUR swap 10s30s** (EUSA30 − EUSA10) and the
**US Treasury 10s30s** (USGG30YR − USGG10YR) are computed against ERZ7 so you
can see the domestic and cross-market relationship side by side.

Everything is done on **daily changes in bp**:

- `10s30s` spread change in bp
- ERZ7 **implied-rate** change in bp (implied rate = 100 − price, so the sign
  convention is rates-space on both legs; a *negative* correlation in this
  convention means the curve **steepens when ERZ7 rallies** — i.e. bull
  steepening)

Outputs: correlation over several windows (daily and weekly changes), OLS
beta / hedge ratio with R², a rolling correlation chart, and a scatter of
daily changes.

**Run all cells, top to bottom. Requires Bloomberg BQuant** — the BQL data
layer cannot run outside the Terminal, so this notebook is untested here;
if anything fails it will be in the single fetch cell below.

In [ ]:
# ---- Config -----------------------------------------------------------------
LOOKBACK   = '-3Y'      # history to pull (relative date)
ROLL_WIN   = 60         # rolling-correlation window (business days)
CORR_WINS  = {'3M': 63, '6M': 126, '1Y': 252, '2Y': 504}   # summary windows
WEEK_STEP  = 5          # non-overlapping weekly changes

TICKERS = {
    'eur10': 'EUSA10 ICPL Index',      # EUR 10y par swap
    'eur30': 'EUSA30 ICPL Index',      # EUR 30y par swap
    'us10' : 'USGG10YR Index',         # UST 10y yield
    'us30' : 'USGG30YR Index',         # UST 30y yield
    'erz7' : 'ERZ7 Comdty',            # 3M Euribor Dec-27 future (price)
}

In [ ]:
# ---- Data layer (the only cell that talks to Bloomberg) ---------------------
import bql
import pandas as pd
import numpy as np

bq = bql.Service()

def fetch_px(tickers, lookback):
    item = {'px': bq.data.px_last(dates=bq.func.range(lookback, '0d'),
                                  fill='prev')}
    resp = bq.execute(bql.Request(list(tickers), item))
    df = resp[0].df().reset_index()
    id_col   = 'ID' if 'ID' in df.columns else 'SECURITY'
    date_col = 'DATE' if 'DATE' in df.columns else 'AS_OF_DATE'
    return (df.pivot_table(index=date_col, columns=id_col, values='px')
              .sort_index())

try:
    raw = fetch_px(TICKERS.values(), LOOKBACK)
    raw.columns.name = None
    PX = raw.rename(columns={v: k for k, v in TICKERS.items()}).dropna(how='all')
    print(f"Fetched {len(PX)} rows, {PX.index[0].date()} -> {PX.index[-1].date()}")
    print(PX.tail(3).round(3))
except Exception as e:
    raise RuntimeError(f"BQL fetch failed: {e}")

In [ ]:
# ---- Compute: levels in bp, daily/weekly changes ----------------------------
LEV = pd.DataFrame({
    'EUR 10s30s': (PX['eur30'] - PX['eur10']) * 100.0,       # bp
    'UST 10s30s': (PX['us30']  - PX['us10'])  * 100.0,       # bp
    'ERZ7 rate' : (100.0 - PX['erz7']) * 100.0,              # implied rate, bp
}).dropna()

CHG_D = LEV.diff().dropna()                     # daily changes, bp
CHG_W = LEV.iloc[::-1].iloc[::WEEK_STEP].iloc[::-1].diff().dropna()  # weekly

SPREADS = ['EUR 10s30s', 'UST 10s30s']

def stats_vs_erz7(chg, spread):
    s, f = chg[spread], chg['ERZ7 rate']
    beta = s.cov(f) / f.var()                   # bp of 10s30s per bp of ERZ7
    corr = s.corr(f)
    return corr, beta, corr**2

rows = []
for spread in SPREADS:
    for label, win in CORR_WINS.items():
        if len(CHG_D) < win:
            continue
        c, b, r2 = stats_vs_erz7(CHG_D.iloc[-win:], spread)
        rows.append({'Spread': spread, 'Window': label, 'Freq': 'daily',
                     'Corr': round(c, 2), 'Beta (bp/bp)': round(b, 2),
                     'R2': round(r2, 2)})
    c, b, r2 = stats_vs_erz7(CHG_W, spread)
    rows.append({'Spread': spread, 'Window': 'full', 'Freq': 'weekly',
                 'Corr': round(c, 2), 'Beta (bp/bp)': round(b, 2),
                 'R2': round(r2, 2)})

SUMMARY = pd.DataFrame(rows)
print("Correlation / beta of 10s30s changes vs ERZ7 implied-rate changes")
print("(negative corr = curve steepens when ERZ7 price rallies)\n")
print(SUMMARY.to_string(index=False))

In [ ]:
# ---- Rolling correlation + scatter ------------------------------------------
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROLL = pd.DataFrame({
    sp: CHG_D[sp].rolling(ROLL_WIN).corr(CHG_D['ERZ7 rate'])
    for sp in SPREADS
}).dropna(how='all')

fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38],
                    subplot_titles=(f'{ROLL_WIN}d rolling corr vs ERZ7 rate',
                                    'Daily changes, last 1Y (bp)'))

for sp in SPREADS:
    fig.add_trace(go.Scatter(x=ROLL.index, y=ROLL[sp], name=sp,
                             mode='lines'), row=1, col=1)
fig.add_hline(y=0, line_dash='dot', line_color='grey', row=1, col=1)

pts = CHG_D.iloc[-252:]
for sp in SPREADS:
    fig.add_trace(go.Scatter(x=pts['ERZ7 rate'], y=pts[sp], mode='markers',
                             name=f'{sp} (scatter)', opacity=0.5,
                             showlegend=False), row=1, col=2)
    b = pts[sp].cov(pts['ERZ7 rate']) / pts['ERZ7 rate'].var()
    a = pts[sp].mean() - b * pts['ERZ7 rate'].mean()
    xs = np.linspace(pts['ERZ7 rate'].min(), pts['ERZ7 rate'].max(), 20)
    fig.add_trace(go.Scatter(x=xs, y=a + b*xs, mode='lines',
                             name=f'{sp} fit ({b:.2f} bp/bp)'), row=1, col=2)

fig.update_xaxes(title_text='ERZ7 implied-rate chg (bp)', row=1, col=2)
fig.update_yaxes(title_text='10s30s chg (bp)', row=1, col=2)
fig.update_layout(height=460, template='plotly_white',
                  legend=dict(orientation='h', y=-0.15))
fig.show()

### Reading the output

- **Corr** — Pearson correlation of changes over the window. In this
  rates-space convention, **negative** means 10s30s steepens when ERZ7
  *price* rallies (front-end rates fall): classic bull-steepening. Flip the
  sign if you think in price terms.
- **Beta (bp/bp)** — bp of 10s30s per 1bp move in ERZ7 implied rate; the
  slope to use for a curve-vs-STIR hedge ratio. Divide risk (DV01s)
  accordingly.
- **Weekly row** — non-overlapping 5-day changes; if it differs a lot from
  daily, the daily relationship is noisy/lead-lagged.
- The rolling chart shows regime shifts — this correlation is very
  regime-dependent (cutting cycles vs term-premium sell-offs), so trust the
  recent windows over the full sample.